[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mihiarc/socialmapper/blob/main/docs/notebooks/07-complete-analysis-workflow.ipynb)

# Complete Analysis Workflow: Food Desert Equity Assessment

## Staunton vs. Waynesboro, Virginia's Shenandoah Valley

This capstone notebook demonstrates a **complete spatial equity analysis** using every core SocialMapper function. We will compare two small neighboring towns in Virginia's Shenandoah Valley -- **Staunton** and **Waynesboro** -- to assess whether they qualify as food deserts, and examine the demographic factors that drive disparities in food access.

Staunton (~25,000 residents) is a historic arts town known for its well-preserved Victorian architecture and thriving downtown. Waynesboro (~23,000 residents), about 15 minutes east, has a more working-class, industrial character. Despite their proximity, these towns have notably different demographic profiles, making them an ideal pair for a food access comparison.

By the end of this notebook, you will know how to:

- Generate walking isochrones to define realistic access areas
- Query OpenStreetMap for food-related points of interest
- Pull Census demographic data (population, income, poverty) for those areas
- Create choropleth maps with POI overlays
- Build comparative bar charts to visualize disparities
- Run a formal multi-location comparison
- Generate a shareable HTML report
- Interpret findings in the context of food justice research

---

## What Is a Food Desert?

The **USDA Economic Research Service** defines a food desert as an area where residents have limited access to affordable, nutritious food. The standard thresholds are:

| Setting | Distance to nearest supermarket |
|---|---|
| **Urban** | > 1 mile (1.6 km) |
| **Rural** | > 10 miles (16 km) |

But distance alone does not tell the whole story. Food deserts are deeply intertwined with structural inequality:

- **Income**: Low-income households spend a larger share of their budget on food and cannot absorb the cost of traveling to distant grocery stores.
- **Vehicle access**: In many food deserts, a significant fraction of households do not own a car. For these residents, *walking distance* is the only meaningful measure of access.
- **Health outcomes**: Research consistently links food desert residence to higher rates of obesity, type 2 diabetes, cardiovascular disease, and other diet-related conditions.
- **Racial disparities**: Food deserts disproportionately affect Black, Latino, and Indigenous communities. A 2021 USDA study found that majority-Black census tracts are twice as likely to be food deserts as majority-white tracts.

Small towns like Staunton and Waynesboro occupy an interesting middle ground in the food desert framework. They are not large urban centers, but they are not truly rural either. Walking access matters here because many lower-income residents still depend on pedestrian infrastructure, and the compact footprint of a small town means that a 15-minute walk can cover a significant portion of the community.

In this analysis, we use **15-minute walking isochrones** rather than simple radius buffers because walking access is the most equitable measure -- it reflects the experience of residents who cannot drive to a grocery store.

> **Why 15 minutes?** Transportation research considers 15 minutes a reasonable one-way walking trip for daily errands. This corresponds to roughly 1 km (0.6 miles), depending on terrain and pedestrian infrastructure.

---

## Town Context

### Staunton, VA

Staunton (pronounced "STAN-ton") is a small city of roughly 25,000 in the heart of Virginia's Shenandoah Valley. It is known for its **historic Wharf district**, a revitalized warehouse area now home to restaurants, galleries, and boutiques. The town also hosts the **American Shakespeare Center's Blackfriars Playhouse**, the world's only recreation of Shakespeare's indoor theater. Staunton has a growing arts and tourism economy, with a walkable downtown that has attracted investment and higher-income newcomers. However, lower-income residents on the outskirts may not share equally in these amenities.

### Waynesboro, VA

Waynesboro sits about 15 minutes east of Staunton, at the base of the Blue Ridge Mountains where the South River runs through town. With a population of roughly 23,000, it has a more **working-class and industrial** heritage, shaped by decades of manufacturing along the river corridor. Waynesboro's downtown along Main Street has seen revitalization efforts, but the town generally has lower incomes and higher poverty rates than Staunton. This economic contrast, combined with geographic proximity, makes the pair an excellent case study for food access disparities.

By comparing these two Shenandoah Valley towns, we can examine how income, geography, and local investment shape food access in small-town America -- a setting often overlooked in food desert research that tends to focus on large cities.

---

## Setup and Imports

In [ ]:
# Uncomment to install on Google Colab:
# !pip install 'socialmapper @ git+https://github.com/mihiarc/socialmapper.git'

In [ ]:
from socialmapper import (
    create_isochrone,
    get_census_blocks,
    get_census_data,
    create_map,
    get_poi,
    analyze_multiple_pois,
    generate_report,
)

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display, HTML

# Consistent plot styling for the entire notebook
plt.rcParams.update({
    "figure.dpi": 150,
    "font.family": "sans-serif",
    "axes.titlesize": 13,
    "axes.labelsize": 11,
})

---

## Step 1: Define the Towns

We use central coordinates for each town. These points serve as the **origin** for isochrone generation and POI searches.

In [ ]:
towns = {
    "Staunton, VA": (38.1496, -79.0717),
    "Waynesboro, VA": (38.0685, -78.8895),
}

for name, (lat, lon) in towns.items():
    print(f"{name}: lat={lat}, lon={lon}")

---

## Step 2: Generate Walking Isochrones

An **isochrone** is a polygon that encloses all the area reachable from a starting point within a given travel time. Unlike a simple circular buffer, isochrones account for the actual road and sidewalk network -- they stretch along walkable corridors and contract around barriers like highways, rivers, and railroad tracks.

We use **walking mode** with a **15-minute** threshold because:

1. Many food desert residents lack vehicle access, making walking the primary mode of transportation for grocery trips.
2. Walking isochrones reveal barriers (highways, rail lines, rivers, industrial zones) that a radius buffer would miss.
3. A 15-minute walk is a widely used threshold in urban planning for "convenient" pedestrian access.
4. In small towns like Staunton and Waynesboro, a 15-minute walk can cover a meaningful portion of the downtown area, making the comparison especially informative.

SocialMapper uses the **Valhalla** open-source routing engine, which models pedestrian routing on the actual OpenStreetMap road network.

In [ ]:
isochrones = {}

for name, coords in towns.items():
    iso = create_isochrone(coords, travel_time=15, travel_mode="walk")
    isochrones[name] = iso
    area = iso["properties"]["area_sq_km"]
    print(f"{name}: {area:.2f} sq km reachable within a 15-minute walk")

print()
staunton_area = isochrones["Staunton, VA"]["properties"]["area_sq_km"]
waynesboro_area = isochrones["Waynesboro, VA"]["properties"]["area_sq_km"]
ratio = staunton_area / waynesboro_area
print(f"Staunton walkable area is {ratio:.2f}x the size of Waynesboro's.")
print("Differences in walkable area reflect street grid density, pedestrian infrastructure, and physical barriers.")

---

## Step 3: Find Food-Related Points of Interest

We search OpenStreetMap for **shopping** (supermarkets, grocery stores, convenience stores) and **food_and_drink** (restaurants, cafes, bakeries) within each walking isochrone.

The `get_poi` function:
1. Creates a walking isochrone behind the scenes to define the search boundary
2. Queries the Overpass API for matching OSM features
3. Computes actual walking travel times via Valhalla's matrix API
4. Returns results sorted by walking time (closest first)

This gives us a realistic picture of what food options a pedestrian resident can reach.

In [ ]:
poi_data = {}

for name, coords in towns.items():
    pois = get_poi(
        coords,
        categories=["shopping", "food_and_drink"],
        travel_time=15,
        travel_mode="walk",
        limit=50,
    )
    poi_data[name] = pois
    print(f"\n{'=' * 50}")
    print(f"{name}: {len(pois)} food/shopping POIs within 15-min walk")
    print(f"{'=' * 50}")
    print(f"{'Name':<35} {'Category':<20} {'Walk (min)'}")
    print(f"{'-'*35} {'-'*20} {'-'*10}")
    for p in pois[:8]:
        travel = p.get("travel_time_minutes", "N/A")
        print(f"{p['name'][:34]:<35} {p['category']:<20} {travel}")

### Interpreting the POI Results

When reading these results, pay attention to:

- **Total count**: More POIs generally means more options, but quality and type matter. A neighborhood with 30 convenience stores and no supermarket is very different from one with 5 full-service grocery stores.
- **Category mix**: Convenience stores often stock processed, shelf-stable items but lack fresh produce. Supermarkets and grocery stores are the gold standard for nutritious food access.
- **Walking times**: Even within the 15-minute isochrone, there is wide variation. A supermarket at 3 minutes is far more accessible than one at 14 minutes, especially for elderly residents or parents with young children.

---

## Step 4: Compare POI Counts (Bar Chart)

A simple bar chart makes the difference in food access immediately visible.

In [ ]:
names = list(towns.keys())
counts = [len(poi_data[n]) for n in names]
colors = ["#4c78a8", "#f58518"]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(names, counts, color=colors, edgecolor="white", linewidth=1.2)

# Add count labels on top of each bar
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.5,
        str(count),
        ha="center", va="bottom", fontweight="bold", fontsize=13,
    )

ax.set_ylabel("Number of Food/Shopping POIs")
ax.set_title("Food Access: POI Count within 15-Minute Walk", fontweight="bold")
ax.set_ylim(0, max(counts) * 1.2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---

## Step 5: Gather Census Demographics

Raw POI counts are only part of the story. To assess **equity**, we need to understand *who lives* in each area. We pull five American Community Survey (ACS) variables:

| Variable | What it tells us |
|---|---|
| `population` | Total residents in each block group |
| `median_income` | Household purchasing power |
| `poverty` | Count of residents below the federal poverty line |
| `housing_units` | Total housing stock (proxy for density) |
| `households_no_vehicle` | Residents who *must* walk or use transit for food access |

The `get_census_data` function fetches data at the **block group** level -- the smallest geography for which the Census Bureau publishes most ACS estimates (typically 600--3,000 people).

In [ ]:
demographic_variables = [
    "population",
    "median_income",
    "poverty",
    "housing_units",
    "households_no_vehicle",
]

blocks_data = {}
census_data = {}
merged_data = {}

for name in towns:
    iso = isochrones[name]

    # Fetch block group boundaries that intersect the isochrone
    blocks = get_census_blocks(polygon=iso)

    # Fetch ACS demographic data for those block groups
    census = get_census_data(iso, variables=demographic_variables)

    # Merge geometry + demographics into a single list of dicts
    merged = []
    for block in blocks:
        geoid = block["geoid"]
        if geoid in census.data:
            merged.append({**block, **census.data[geoid]})

    blocks_data[name] = blocks
    census_data[name] = census
    merged_data[name] = merged

    # Summarize
    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    print(f"\n{name}:")
    print(f"  Block groups matched: {len(merged)}")
    print(f"  Total population:     {total_pop:,}")

---

## Step 6: Demographic Summary Table

Let us compute aggregate statistics for each town and display them in a clean comparison table.

In [ ]:
summary = {}

for name in towns:
    census = census_data[name]
    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rate = (total_poverty / total_pop * 100) if total_pop > 0 else 0
    no_vehicle = sum(
        d.get("households_no_vehicle", 0)
        for d in census.data.values()
        if d.get("households_no_vehicle") is not None
    )
    total_housing = sum(
        d.get("housing_units", 0)
        for d in census.data.values()
        if d.get("housing_units") is not None
    )

    summary[name] = {
        "Population": total_pop,
        "Avg. Median Income": f"${avg_income:,.0f}",
        "Poverty Count": total_poverty,
        "Poverty Rate": f"{poverty_rate:.1f}%",
        "Households w/o Vehicle": no_vehicle,
        "Housing Units": total_housing,
        "Walkable Area (sq km)": f"{isochrones[name]['properties']['area_sq_km']:.2f}",
        "Food/Shopping POIs": len(poi_data[name]),
    }

summary_df = pd.DataFrame(summary)
display(summary_df)

---

## Step 7: Grouped Bar Chart -- Demographic Comparison

Visualizing population, income, and poverty side by side helps identify structural differences between the two towns.

In [ ]:
# Collect numeric values for the grouped bar chart
demo_metrics = {}
for name in towns:
    census = census_data[name]
    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    demo_metrics[name] = {
        "Population": total_pop,
        "Avg. Median Income ($)": avg_income,
        "Poverty Count": total_poverty,
    }

metric_names = list(demo_metrics["Staunton, VA"].keys())
staunton_vals = [demo_metrics["Staunton, VA"][m] for m in metric_names]
waynesboro_vals = [demo_metrics["Waynesboro, VA"][m] for m in metric_names]

import numpy as np

x = np.arange(len(metric_names))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width / 2, staunton_vals, width, label="Staunton, VA", color="#4c78a8", edgecolor="white")
bars2 = ax.bar(x + width / 2, waynesboro_vals, width, label="Waynesboro, VA", color="#f58518", edgecolor="white")

# Add value labels
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        label = f"{height:,.0f}"
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height + max(staunton_vals + waynesboro_vals) * 0.01,
            label, ha="center", va="bottom", fontsize=9,
        )

ax.set_xticks(x)
ax.set_xticklabels(metric_names)
ax.set_title("Demographic Comparison: Staunton vs. Waynesboro", fontweight="bold")
ax.legend()
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

---

## Step 8: Population Choropleth Maps with POI Overlays

Choropleth maps color each block group by a numeric variable. Overlaying POI markers on top reveals how food options are distributed relative to population density.

Look for:
- **Clusters of POIs** vs. **gaps** in coverage
- Whether high-population areas are well-served or underserved
- How the isochrone boundary (dashed line) relates to the block group edges

In [ ]:
for name in towns:
    overlay_points = [
        {"lat": p["lat"], "lon": p["lon"], "name": p["name"]}
        for p in poi_data[name][:15]
    ]

    map_result = create_map(
        data=merged_data[name],
        column="population",
        title=f"Population by Block Group -- {name}",
        overlay_boundary=isochrones[name],
        overlay_points=overlay_points,
        show_stats=True,
    )
    print(f"\n--- {name} ---")
    display(Image(data=map_result.image_data))

### Reading the Population Maps

Each block group is shaded by total population -- darker areas are more densely populated. The **magenta dots** represent food and shopping POIs, while the **dashed boundary** marks the 15-minute walking isochrone.

Key questions to consider:
- Are the most populated block groups (darkest shading) well-covered by POI markers?
- Are there any high-population areas with no nearby food options?
- How does the street layout differ between the two towns? Staunton's historic grid may yield a different walkable shape than Waynesboro's more linear river-corridor layout.

---

## Step 9: Income Choropleth Maps

Income maps reveal economic stratification within each walking area. We use the `RdYlGn` (red-yellow-green) colormap so that low-income areas appear in red and high-income areas in green -- a natural mapping for "financial health."

In [ ]:
for name in towns:
    income_map = create_map(
        data=merged_data[name],
        column="median_income",
        title=f"Median Household Income -- {name}",
        overlay_boundary=isochrones[name],
        show_stats=True,
        cmap="RdYlGn",
    )
    print(f"\n--- {name} ---")
    display(Image(data=income_map.image_data))

### Reading the Income Maps

Look for the spatial distribution of income within each town:

- **Staunton** may show higher incomes near the revitalized Wharf district and historic downtown, with lower incomes in outlying residential areas.
- **Waynesboro** may show generally lower income levels, with variation tied to proximity to the Main Street corridor and industrial areas along the South River.

Remember: `median_income` is reported per block group. A single block group's median can mask wide variation among individual households.

---

## Step 10: Poverty Rate Comparison

The poverty rate (percentage of residents below the federal poverty line) is one of the strongest predictors of food desert status. Let us compare it directly.

In [ ]:
poverty_rates = {}
for name in towns:
    census = census_data[name]
    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rates[name] = (total_poverty / total_pop * 100) if total_pop > 0 else 0

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    list(poverty_rates.keys()),
    list(poverty_rates.values()),
    color=["#e45756", "#f58518"],
    edgecolor="white",
    linewidth=1.2,
)

for bar, rate in zip(bars, poverty_rates.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{rate:.1f}%",
        ha="center", va="bottom", fontweight="bold", fontsize=13,
    )

ax.set_ylabel("Poverty Rate (%)")
ax.set_title("Poverty Rate Comparison (15-min Walk Area)", fontweight="bold")
ax.set_ylim(0, max(poverty_rates.values()) * 1.35)
ax.axhline(y=20, color="gray", linestyle="--", linewidth=0.8, alpha=0.6)
ax.text(1.02, 20, "20% threshold", va="center", ha="left", fontsize=9, color="gray", transform=ax.get_yaxis_transform())
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

print("Note: The USDA considers a 20% poverty rate as a key threshold for food desert classification.")

---

## Step 11: POIs per Capita -- The Equity Metric

Raw POI counts do not account for how many people those POIs must serve. A neighborhood with 40 POIs and 20,000 residents has very different per-capita access than one with 40 POIs and 5,000 residents.

**POIs per 1,000 residents** normalizes the comparison and is a better measure of food access equity.

In [ ]:
per_capita = {}
for name in towns:
    census = census_data[name]
    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    n_pois = len(poi_data[name])
    rate = (n_pois / total_pop * 1000) if total_pop > 0 else 0
    per_capita[name] = rate

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(
    list(per_capita.keys()),
    list(per_capita.values()),
    color=["#4c78a8", "#f58518"],
    edgecolor="white",
    linewidth=1.2,
)

for bar, rate in zip(bars, per_capita.values()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.05,
        f"{rate:.1f}",
        ha="center", va="bottom", fontweight="bold", fontsize=13,
    )

ax.set_ylabel("POIs per 1,000 Residents")
ax.set_title("Food Access per Capita (15-min Walk Area)", fontweight="bold")
ax.set_ylim(0, max(per_capita.values()) * 1.3)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

higher = max(per_capita, key=per_capita.get)
lower = min(per_capita, key=per_capita.get)
ratio = per_capita[higher] / per_capita[lower] if per_capita[lower] > 0 else float("inf")
print(f"{higher} has {ratio:.1f}x the per-capita food access of {lower}.")

---

## Step 12: Formal Multi-Location Comparison

The `analyze_multiple_pois` function performs a structured, side-by-side analysis of multiple locations. It:

1. Creates isochrones for each location (in parallel)
2. Pulls census data for the specified variables
3. Aggregates totals, means, mins, and maxes
4. Ranks locations by each variable

This is the same analysis we did manually above, but packaged into a single function call -- useful for programmatic comparisons across many towns or locations.

In [ ]:
comparison = analyze_multiple_pois(
    locations=list(towns.values()),
    travel_time=15,
    travel_mode="walk",
    variables=["population", "median_income", "poverty", "housing_units"],
)

print("=" * 65)
print("MULTI-LOCATION COMPARISON RESULTS")
print("=" * 65)

for var, info in comparison["comparison"].items():
    print(f"\n--- {var.upper()} ---")
    print(f"  {'Location':<28} {'Total':>12}  {'Mean':>10}")
    for rank in info["ranked"]:
        print(f"  {rank['location']:<28} {rank['total']:>12,.0f}  {rank['mean']:>10,.0f}")
    print(f"  Highest: {info['highest']}")
    print(f"  Lowest:  {info['lowest']}")

---

## Step 13: Generate HTML Report

The `generate_report` function transforms the comparison dictionary into a formatted, shareable HTML document. This is useful for distributing findings to stakeholders who may not use Jupyter notebooks.

In [ ]:
report_html = generate_report(comparison, format="html")
print(f"Generated HTML report: {len(report_html):,} characters")
display(HTML(report_html))

---

## Key Findings

Let us consolidate all the metrics we have computed into a final summary and interpret what they mean for food access equity.

In [ ]:
print("=" * 65)
print("FOOD ACCESS EQUITY ASSESSMENT")
print("Shenandoah Valley: Staunton vs. Waynesboro (15-min walk)")
print("=" * 65)

findings = {}

for name in towns:
    census = census_data[name]
    pois = poi_data[name]
    iso = isochrones[name]

    total_pop = sum(
        d.get("population", 0)
        for d in census.data.values()
        if d.get("population") is not None
    )
    incomes = [
        d["median_income"]
        for d in census.data.values()
        if d.get("median_income") is not None and d["median_income"] > 0
    ]
    avg_income = sum(incomes) / len(incomes) if incomes else 0
    total_poverty = sum(
        d.get("poverty", 0)
        for d in census.data.values()
        if d.get("poverty") is not None
    )
    poverty_rate = (total_poverty / total_pop * 100) if total_pop > 0 else 0
    pois_per_1k = (len(pois) / total_pop * 1000) if total_pop > 0 else 0
    no_vehicle = sum(
        d.get("households_no_vehicle", 0)
        for d in census.data.values()
        if d.get("households_no_vehicle") is not None
    )

    findings[name] = {
        "population": total_pop,
        "avg_income": avg_income,
        "poverty_rate": poverty_rate,
        "n_pois": len(pois),
        "pois_per_1k": pois_per_1k,
        "area": iso["properties"]["area_sq_km"],
        "no_vehicle": no_vehicle,
    }

    print(f"\n--- {name} ---")
    print(f"  Walkable area:           {iso['properties']['area_sq_km']:.2f} sq km")
    print(f"  Population:              {total_pop:,}")
    print(f"  Avg. median income:      ${avg_income:,.0f}")
    print(f"  Poverty rate:            {poverty_rate:.1f}%")
    print(f"  Households w/o vehicle:  {no_vehicle:,}")
    print(f"  Food/shopping POIs:      {len(pois)}")
    print(f"  POIs per 1,000 people:   {pois_per_1k:.1f}")

### Interpretation

The numbers above tell a nuanced story. Here are the key takeaways:

1. **POI count alone is misleading.** One town may have more raw POIs, but once you normalize by population, the per-capita picture can look very different. Always compute POIs per 1,000 residents.

2. **Poverty rate matters.** A town with a high poverty rate and low per-capita food access faces a double burden: residents have less money to spend on food *and* fewer places to spend it. If either town exceeds the 20% poverty threshold, it meets one of the USDA's food desert criteria.

3. **Vehicle access is the hidden variable.** Households without vehicles are entirely dependent on walking-distance food options. A town with many car-free households and few nearby grocery stores is functionally a food desert for its most vulnerable residents, even if a supermarket exists a short drive away.

4. **Walkable area reflects infrastructure.** Differences in isochrone size reveal differences in pedestrian infrastructure. A larger walkable area means more interconnected streets and fewer barriers -- a sign of a more walkable built environment. Staunton's historic street grid and Waynesboro's river-corridor layout may produce noticeably different walkable footprints.

5. **Income disparities compound access issues.** Lower-income residents are less able to pay for delivery services, rideshares to distant grocery stores, or premium prices at nearby convenience stores. Income and access interact multiplicatively, not additively.

6. **Small-town dynamics differ from big-city patterns.** Unlike large urban neighborhoods where a food desert may exist amid surrounding abundance, small Shenandoah Valley towns have fewer alternatives. If the walkable core lacks grocery options, there may be no nearby transit or dense commercial corridor to compensate.

---

## Limitations and Next Steps

Every analysis has limitations. Acknowledging them is essential for responsible use of the results.

### Data Limitations

- **OpenStreetMap completeness**: OSM is crowd-sourced, and coverage varies by location. Small towns in the Shenandoah Valley may have less complete OSM coverage than major metro areas, meaning some stores could be missing while others may be listed but closed. This bias can make food deserts appear *better served* than they actually are.

- **ACS margins of error**: Census data at the block group level comes from the American Community Survey 5-year estimates, which are *sample-based* and carry significant margins of error, especially for small populations. In towns of 23,000--25,000 people, block group estimates should be treated with extra caution.

- **Isochrone limitations**: Walking isochrones assume a healthy adult walking at average speed on clear sidewalks. They do not account for:
  - Safety concerns (crime, poor lighting, lack of crosswalks)
  - Weather (Shenandoah Valley winters and summer heat can make walking more difficult)
  - Terrain (both towns have hilly areas that slow walking speeds)
  - Mobility limitations (elderly, disabled, parents with strollers)
  - Carrying capacity (a 15-minute walk is much harder with bags of groceries)

- **POI categories**: Our search uses broad categories ("shopping", "food_and_drink"). A convenience store selling chips and soda is counted the same as a full-service supermarket with fresh produce. Future work should distinguish between these.

### Suggested Extensions

- **More Valley towns**: Add additional Shenandoah Valley communities (e.g., Harrisonburg, Lexington, Buena Vista) to `analyze_multiple_pois` for a regional comparison.
- **Vehicle access analysis**: Cross-reference `households_no_vehicle` with POI density to identify areas where car-free households have the worst food access.
- **Driving isochrones**: Compare walking vs. driving isochrones to quantify the "car advantage" in food access -- especially relevant in small towns where most residents drive.
- **Time series**: Pull ACS data for multiple years to track whether food access is improving or declining.
- **Custom POI data**: Use `import_poi_csv()` to incorporate local health department food inspection data, which may be more complete than OSM.
- **Interactive maps**: Use `create_map(..., export_format='html')` for zoomable, clickable maps that stakeholders can explore.

---

## API Cheat Sheet

Quick reference for all SocialMapper functions used in this notebook:

| Function | Purpose | Key Parameters |
|---|---|---|
| `create_isochrone(location, travel_time, travel_mode)` | Travel-time polygon from a point | `travel_mode`: `"walk"`, `"drive"`, `"bike"` |
| `get_census_blocks(polygon=iso)` | Census block groups intersecting an area | Pass an isochrone or GeoJSON dict |
| `get_census_data(location, variables)` | ACS demographic data by block group | Variables: `"population"`, `"median_income"`, `"poverty"`, etc. |
| `create_map(data, column, ...)` | Choropleth map with optional overlays | `overlay_boundary`, `overlay_points`, `show_stats`, `cmap` |
| `get_poi(location, categories, ...)` | OpenStreetMap points of interest | `travel_time` triggers isochrone-bounded search |
| `analyze_multiple_pois(locations, ...)` | Multi-location demographic comparison | Returns rankings for each variable |
| `generate_report(data, format)` | Formatted HTML report from analysis data | `format`: `"html"` |
| `import_poi_csv(path)` | Custom POI data from CSV file | Specify column mappings for lat/lon/name |

---

*This notebook was created as part of the SocialMapper tutorial series. For more information, see the [SocialMapper documentation](https://github.com/mihiarc/socialmapper).*